# Engineered features

Create leakage-safe engineered features from the preprocessed artifacts produced by notebook 03. The output of this notebook is a new fold pickle plus matching engineered train/test CSVs for downstream single-model and stacking experiments.

Notebook 04 and earlier remain unchanged.

## 1. Notebook setup

### 1.1. Imports

In [1]:
import pickle

import numpy as np
import pandas as pd

### 1.2. Run configuration

In [2]:
ENGINEERED_CV_FOLDS = '../data/tmp/08-engineered-cv-folds.pkl'
ENGINEERED_TRAIN_DATA = '../data/tmp/08-engineered-train-data.csv'
ENGINEERED_TEST_DATA = '../data/tmp/08-engineered-test-data.csv'

SOURCE_CV_FOLDS = '../data/tmp/03-preprocessed-cv-folds.pkl'
SOURCE_TRAIN_DATA = '../data/tmp/03-preprocessed-train-data.csv'
SOURCE_TEST_DATA = '../data/tmp/03-preprocessed-test-data.csv'

SLEEP_BINS = [0, 5, 7, 9, np.inf]
BMI_BINS = [0, 18.5, 25.0, 30.0, np.inf]
STEP_BINS = [0, 2500, 5000, 7500, np.inf]
EXERCISE_BINS = [0, 15, 30, 60, np.inf]
WATER_BINS = [0, 1, 2, 3, np.inf]

## 2. Load artifacts from notebook 03

In [3]:
with open(SOURCE_CV_FOLDS, 'rb') as handle:
    cv_folds = pickle.load(handle)

train_df = pd.read_csv(SOURCE_TRAIN_DATA)
test_df = pd.read_csv(SOURCE_TEST_DATA)

base_feature_columns = [column for column in train_df.columns if column != 'health_condition']
missing_indicator_columns = [column for column in base_feature_columns if column.endswith('_missing')]

print(f'Loaded {len(cv_folds)} folds')
print(f'Preprocessed train shape: {train_df.shape}')
print(f'Preprocessed test shape:  {test_df.shape}')
print(f'Missing indicator columns: {missing_indicator_columns}')

Loaded 10 folds
Preprocessed train shape: (690088, 33)
Preprocessed test shape:  (295753, 33)
Missing indicator columns: ['sleep_duration_missing', 'heart_rate_missing', 'bmi_missing', 'calorie_expenditure_missing', 'step_count_missing', 'exercise_duration_missing', 'water_intake_missing']


## 3. Build engineered features

In [ ]:
def safe_divide(numerator, denominator):

    denominator = denominator.replace(0, np.nan)
    result = numerator / denominator

    return result.replace([np.inf, -np.inf], np.nan).fillna(0.0)


def add_engineered_features(df):

    engineered = df.copy()

    engineered['step_rate'] = safe_divide(engineered['step_count'], engineered['exercise_duration'])
    engineered['calorie_rate'] = safe_divide(engineered['calorie_expenditure'], engineered['exercise_duration'])
    engineered['water_rate'] = safe_divide(engineered['water_intake'], engineered['exercise_duration'])
    engineered['sleep_to_activity'] = safe_divide(engineered['sleep_duration'], engineered['exercise_duration'])
    engineered['activity_intensity'] = safe_divide(engineered['step_count'], engineered['exercise_duration'] + 1.0)
    engineered['calorie_density'] = safe_divide(engineered['calorie_expenditure'], engineered['bmi'] + 1.0)
    engineered['recovery_ratio'] = safe_divide(engineered['sleep_duration'], engineered['heart_rate'] + 1.0)

    if missing_indicator_columns:
        engineered['missing_indicator_count'] = engineered[missing_indicator_columns].sum(axis=1)

    else:
        engineered['missing_indicator_count'] = 0.0

    engineered['sleep_duration_band'] = pd.cut(
        engineered['sleep_duration'],
        bins=SLEEP_BINS,
        labels=False,
        include_lowest=True,
    ).fillna(-1).astype(int)

    engineered['bmi_band'] = pd.cut(
        engineered['bmi'],
        bins=BMI_BINS,
        labels=False,
        include_lowest=True,
    ).fillna(-1).astype(int)

    engineered['step_count_band'] = pd.cut(
        engineered['step_count'],
        bins=STEP_BINS,
        labels=False,
        include_lowest=True,
    ).fillna(-1).astype(int)

    engineered['exercise_duration_band'] = pd.cut(
        engineered['exercise_duration'],
        bins=EXERCISE_BINS,
        labels=False,
        include_lowest=True,
    ).fillna(-1).astype(int)

    engineered['water_intake_band'] = pd.cut(
        engineered['water_intake'],
        bins=WATER_BINS,
        labels=False,
        include_lowest=True,
    ).fillna(-1).astype(int)

    return engineered


engineered_folds = []

for fold in cv_folds:
    engineered_fold = fold.copy()
    engineered_fold['x_train'] = add_engineered_features(fold['x_train'])
    engineered_fold['x_validation'] = add_engineered_features(fold['x_validation'])
    engineered_folds.append(engineered_fold)

engineered_train_df = add_engineered_features(train_df)
engineered_test_df = add_engineered_features(test_df)

print(f'Engineered train shape: {engineered_train_df.shape}')
print(f'Engineered test shape:  {engineered_test_df.shape}')
print(f'Added engineered columns: {[column for column in engineered_train_df.columns if column not in train_df.columns]}')

Engineered train shape: (690088, 46)
Engineered test shape:  (295753, 46)
Added engineered columns: ['step_rate', 'calorie_rate', 'water_rate', 'sleep_to_activity', 'activity_intensity', 'calorie_density', 'recovery_ratio', 'missing_indicator_count', 'sleep_duration_band', 'bmi_band', 'step_count_band', 'exercise_duration_band', 'water_intake_band']


## 4. Save engineered artifacts

In [5]:
with open(ENGINEERED_CV_FOLDS, 'wb') as handle:
    pickle.dump(engineered_folds, handle)

engineered_train_df.to_csv(ENGINEERED_TRAIN_DATA, index=False)
engineered_test_df.to_csv(ENGINEERED_TEST_DATA, index=False)

print('Saved engineered artifacts:')
print(f'  {ENGINEERED_CV_FOLDS}')
print(f'  {ENGINEERED_TRAIN_DATA}')
print(f'  {ENGINEERED_TEST_DATA}')

Saved engineered artifacts:
  ../data/tmp/08-engineered-cv-folds.pkl
  ../data/tmp/08-engineered-train-data.csv
  ../data/tmp/08-engineered-test-data.csv
